# EvoHash — анализ эволюционного прогона
Выбери хэш, ран и бин — ноутбук покажет историю программ и позволит извлечь нужную рядом с собой.

In [ ]:
import json
import shutil
from pathlib import Path

# В Jupyter __file__ не определён; CWD — папка где лежит ноутбук
NOTEBOOK_DIR  = Path().resolve()
REPO_ROOT     = NOTEBOOK_DIR.parent
SNAPSHOTS_DIR = REPO_ROOT / "snapshots"

# ── НАСТРОЙКИ ──────────────────────────────────────────────────────────────────
PHF    = "phash"   # phash | pdq | neuralhash | photodna
RUN_ID = "latest"  # конкретный run_YYYYMMDD_HHMMSS или "latest"
# ───────────────────────────────────────────────────────────────────────────────

phf_dir = SNAPSHOTS_DIR / PHF
assert phf_dir.exists(), f"Папка {phf_dir} не найдена"

runs = sorted([d for d in phf_dir.iterdir() if d.is_dir()], reverse=True)
assert runs, f"Нет ранов в {phf_dir}"

if RUN_ID == "latest":
    run_dir = runs[0]
else:
    run_dir = phf_dir / RUN_ID
    assert run_dir.exists(), f"Run {RUN_ID} не найден"

print(f"PHF   : {PHF}")
print(f"Run   : {run_dir.name}")
print()
print(f"Все раны для {PHF}:")
for r in runs:
    h = {}
    if (r / 'history.json').exists():
        try:
            h = json.loads((r / 'history.json').read_text('utf-8'))
        except Exception:
            pass
    bins = h.get('bins', {})
    n = sum(len(v) for v in bins.values())
    snap = '  [redis snapshot]' if (r / 'redis_snapshot.json').exists() else ''
    marker = ' <-- выбран' if r == run_dir else ''
    print(f"  {r.name}  — {len(bins)} бинов, {n} программ{snap}{marker}")

In [ ]:
history_file = run_dir / "history.json"
assert history_file.exists(), "history.json не найден — ран ещё не записан или пустой"

history = json.loads(history_file.read_text("utf-8"))
bins = history["bins"]

print(f"Бинов в архиве: {len(bins)}\n")
print(f"  {'Бин':<12} {'Программ':>9}  {'Лучший eff':>12}  Лучшая программа")
print("  " + "-" * 62)

for cell_key, entries in sorted(bins.items(), key=lambda x: x[0]):
    best = max(entries, key=lambda e: e.get("efficiency") or -1)
    best_eff = best.get("efficiency")
    eff_str = f"{best_eff:.6f}" if best_eff is not None else "—"
    print(f"  {cell_key:<12} {len(entries):>9}  {eff_str:>12}  {best['name']}")

In [ ]:
# ── ВЫБЕРИ БИН ─────────────────────────────────────────────────────────────────
CELL = "0"   # cell key из таблицы выше, например "0" или "3,5"
# ───────────────────────────────────────────────────────────────────────────────

assert CELL in bins, f"Бин '{CELL}' не найден. Доступные: {sorted(bins.keys())}"

entries = bins[CELL]

print(f"История бина '{CELL}'  ({len(entries)} программ, в порядке добавления):\n")
print(f"  {'#':>3}  {'Название':<36}  {'Efficiency':>12}  {'ASR':>6}  {'L2':>8}")
print("  " + "-" * 72)

best_eff = max((e.get('efficiency') or -1) for e in entries)
for i, e in enumerate(entries):
    eff = e.get('efficiency')
    asr = e.get('ASR')
    l2  = e.get('L2')
    eff_s = f"{eff:.6f}" if eff is not None else "—"
    asr_s = f"{asr:.4f}" if asr is not None else "—"
    l2_s  = f"{l2:.3f}"  if l2  is not None else "—"
    marker = "  <-- best" if eff == best_eff else ""
    print(f"  {i:>3}  {e['name']:<36}  {eff_s:>12}  {asr_s:>6}  {l2_s:>8}{marker}")

In [ ]:
import matplotlib.pyplot as plt

effs = [e.get('efficiency') for e in entries]
valid = [(i, e) for i, e in enumerate(effs) if e is not None]

if valid:
    xs, ys = zip(*valid)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(xs, ys, marker='o', markersize=4, linewidth=1.2, color='#39ff14')
    ax.fill_between(xs, ys, alpha=0.15, color='#39ff14')
    ax.set_title(f"Efficiency в бине '{CELL}' по мере заполнения")
    ax.set_xlabel("порядковый номер программы")
    ax.set_ylabel("efficiency")
    plt.tight_layout()
    plt.show()
else:
    print("Нет данных efficiency для графика")

In [ ]:
# ── ИЗВЛЕЧЬ ПРОГРАММУ ──────────────────────────────────────────────────────────
# Укажи индекс из таблицы выше, "best" для лучшей по efficiency, или "last"
EXTRACT = "best"   # int | "best" | "last"
# ───────────────────────────────────────────────────────────────────────────────

if EXTRACT == "best":
    idx = max(range(len(entries)), key=lambda i: entries[i].get('efficiency') or -1)
elif EXTRACT == "last":
    idx = len(entries) - 1
else:
    idx = int(EXTRACT)

entry = entries[idx]
name  = entry['name']

bin_folder = "bin_" + CELL.replace(",", "_")
src_py = run_dir / bin_folder / f"{name}.py"
assert src_py.exists(), f"Файл {src_py} не найден — возможно, ран был записан старой версией без .py"

# Удалить старые извлечённые файлы рядом с ноутбуком
for old in NOTEBOOK_DIR.glob("extracted_program_*.py"):
    old.unlink()
    print(f"Удалён: {old.name}")

dst_name = f"extracted_program_{PHF}_{run_dir.name}_bin{CELL.replace(',','_')}_{name}.py"
dst_py = NOTEBOOK_DIR / dst_name
shutil.copy2(src_py, dst_py)

print(f"Извлечена программа #{idx}: {name}")
print(f"  Efficiency : {entry.get('efficiency')}")
print(f"  ASR        : {entry.get('ASR')}")
print(f"  L2         : {entry.get('L2')}")
print(f"  Файл       : {dst_py.name}")

In [ ]:
# Предпросмотр кода
print(dst_py.read_text('utf-8'))